<p> <center><img src="img.jpg" style="height:200px"></center> </p>

<hr style="border-width:2px;border-color:#AAD">
<center><h1> Analyse et prédiction de la variabilité de la production solaire à partir de données ouvertes de la région PACA </h1></center>
<center><h2> Collecte des données astronomiques </h2></center>
<hr style="border-width:2px;border-color:#AAD">


Nous avons précédemment collecté les données de **production d'énergie solaire**, contenant la base de calcul de notre variable cible. Or, les autres variables du jeu de données collecté ne permettent à priori pas d'expliquer la variable cible.

 - Elles ne contiennent pas de données sur la **position du soleil** par rapport aux panneaux solaires ;
 - Elles ne contiennent pas de données **météorologiques**.

Pour déterminer quelles données pourraient permettre d'expliquer notre variable cible et pourraient être utiles à notre modélisation, nous sommes partis des connaissances usuelles de l'homme du métier (voir les articles retenus).

Parmi ces connaissances, on a constaté que la position du soleil par rapport à l'emplacement des panneaux solaires était importante et que cette position était déterminée par l'**azimut** et l'**angle solaire**.

On a vu que la région PACA était vaste et on a déterminé **cinq lieux significatifs** pour la production d'énergie solaire : nous allons calculer l'azimut et l'angle solaire pour chacun d'eux.

# I - Récupération des données de localisation et de production


Nous commencons par importer les librairies nécessaires pour manipuler nos données :

In [1]:
# Gestion des chemins
from pathlib import Path

# Traitement des datasets
import pandas as pd
import numpy as np

On récupère les données de production collectées aux étapes précédentes depuis le répertoire du projet.

In [2]:
# Chemin vers le répertoire de résultats temporaires
temp_path = Path('../data/local_data/temp/')

# Chemin vers le répertoire de résultats finaux
output_path = Path('../data/local_data/output/')

# Chemin du dataset de production
input_datasets = temp_path / 'production_2020_2025.csv'

# Chemin du dataset des communes sélectionnées 
communes = output_path / 'best_communes_geo_energy.csv'


On charge le jeu de données de l'étape précédente :

In [3]:
# Charger le dataset production final de l'étape précédente :
df_astro = pd.read_csv(input_datasets, parse_dates=['datetime_utc'])

# Pour la collecte des données on n'a besoin que des créneaux horaires
df_astro = df_astro[['datetime_utc']].copy() 
display(df_astro.head())


,datetime_utc
0,2019-12-31 23:00:00+00:00
1,2019-12-31 23:30:00+00:00
2,2020-01-01 00:00:00+00:00
3,2020-01-01 00:30:00+00:00
4,2020-01-01 01:00:00+00:00


# II - Lieux retenus

On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.


In [4]:
df_communes = pd.read_csv(communes)
display(df_communes.head())
print(df_communes.dtypes)

,cluster_geo,best_commune,code_insee,lat,lon,energie_totale,poids,prefix
0,2,Cruis,4065,44.0845,5.8397,20356525.0,0.22,cru
1,4,Saint-Étienne-le-Laus,5140,44.5075,6.1616,325158.0,0.06,sel
2,0,Saint-Vallier-de-Thiey,6130,43.6994,6.8516,344281.0,0.07,svt
3,1,Bras,83021,43.4723,5.9558,10603661.0,0.29,bra
4,3,Eygalières,13034,43.7638,4.9554,1510927.0,0.36,eyg


cluster_geo         int64
best_commune       object
code_insee          int64
lat               float64
lon               float64
energie_totale    float64
poids             float64
prefix             object
dtype: object


# III - Calcul de l'azimut et de l'altitude

Des passionnés d'astronomie ont déjà mis à dispositions plusieurs librairies permettant de calculer l'azimut et l'altitude du soleil (= angle solaire).

On va utiliser la librairie PySolar : https://pysolar.readthedocs.io/en/latest/.

In [5]:
# On importe les librairies dont on a besoin
from pysolar.solar import *
import datetime

Enfin on utilise la librairie pysolar pour calculer l'azimuth et l'angle solaire au niveau des cinq points significatifs retenus pour chaque observation de notre jeu de données.

In [6]:
# Attention, l'exécution de cette cellule est longue !
# On parcours les villes
for ville in df_communes['prefix'] :
    print(ville + "...")
    latitude = df_communes.loc[df_communes['prefix']==ville, 'lat'].iloc[0]
    longitude = df_communes.loc[df_communes['prefix']==ville, 'lon'].iloc[0]

    # On crée une nouvelle colonne pour la ville pour l'azimuth
    print("\tAzimuth...")
    df_astro[ville + '_azimuth'] = df_astro['datetime_utc'].apply(lambda date : get_azimuth(latitude, longitude, date))
    print("\t\tOK")

    # On crée une nouvelle colonne pour la ville pour l'altitude
    print("\tAltitude...")
    df_astro[ville + '_altitude'] = df_astro['datetime_utc'].apply(lambda date : get_altitude(latitude, longitude, date))
    print("\t\tOK")

    print(ville + "-> OK\n")
    

cru...
	Azimuth...
		OK
	Altitude...
		OK
cru-> OK

sel...
	Azimuth...
		OK
	Altitude...
		OK
sel-> OK

svt...
	Azimuth...
		OK
	Altitude...
		OK
svt-> OK

bra...
	Azimuth...
		OK
	Altitude...
		OK
bra-> OK

eyg...
	Azimuth...
		OK
	Altitude...
		OK
eyg-> OK



On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [7]:
display(df_astro.head(10))

,datetime_utc,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,bra_azimuth,bra_altitude,eyg_azimuth,eyg_altitude
0,2019-12-31 23:00:00+00:00,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709
5,2020-01-01 01:30:00+00:00,56.104984,-59.129358,55.961399,-58.701947,58.001012,-58.726843,57.122520,-59.396257,55.332468,-59.836248
6,2020-01-01 02:00:00+00:00,65.346368,-54.428472,65.171692,-54.042195,66.944253,-53.917283,66.258196,-54.602951,64.764317,-55.141210
7,2020-01-01 02:30:00+00:00,73.008055,-49.393849,72.842496,-49.049410,74.372636,-48.802125,73.801849,-49.488368,72.533553,-50.097161
8,2020-01-01 03:00:00+00:00,79.584354,-44.161912,79.447862,-43.858394,80.769679,-43.509092,80.264541,-44.186485,79.165262,-44.847029
9,2020-01-01 03:30:00+00:00,85.424299,-38.823177,85.326319,-38.559425,86.472049,-38.123002,86.000566,-38.785359,85.027894,-39.484567


# IV - Enregistrement des données obtenues de Pysolar

Nous avons terminé la collecte des données explicatives relatives à l'`azimuth` et à l'`angle solaire` pour chaque point significatif du point de vue de la production d'énergie solaire, grâce à la librairie **PySolar**.

Nous enregistrons notre jeu de données actuel pour clore la deuxième partie de notre travail de collecte.


In [8]:
# On enregistre ce dataset contenant les données de production et les données astronomiques avant ajout d'autres variables
df_astro.to_csv(temp_path / "astronomie_2020_2025.csv", index=False)


In [9]:
# Exemple de la manière dont charger ce dataset final de données astronomiques :
df = pd.read_csv(temp_path / "astronomie_2020_2025.csv", parse_dates=['datetime_utc'])
display(df.head())
df.info()


,datetime_utc,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,bra_azimuth,bra_altitude,eyg_azimuth,eyg_altitude
0,2019-12-31 23:00:00+00:00,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106704 entries, 0 to 106703
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   datetime_utc  106704 non-null  datetime64[ns, UTC]
 1   cru_azimuth   106704 non-null  float64            
 2   cru_altitude  106704 non-null  float64            
 3   sel_azimuth   106704 non-null  float64            
 4   sel_altitude  106704 non-null  float64            
 5   svt_azimuth   106704 non-null  float64            
 6   svt_altitude  106704 non-null  float64            
 7   bra_azimuth   106704 non-null  float64            
 8   bra_altitude  106704 non-null  float64            
 9   eyg_azimuth   106704 non-null  float64            
 10  eyg_altitude  106704 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(10)
memory usage: 9.0 MB


In [10]:
df.describe()

,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,bra_azimuth,bra_altitude,eyg_azimuth,eyg_altitude
count,106704.000000,106704.000000,106704.000000,106704.000000,106704.000000,106704.000000,106704.000000,106704.000000,106704.000000,106704.000000
mean,180.405747,0.096757,180.592687,0.097105,180.385269,0.096398,180.474625,0.096387,179.889195,0.096386
std,101.204852,34.081557,101.259135,33.864242,101.159647,34.279100,101.125068,34.395467,101.165752,34.246108
min,0.017255,-69.332104,0.001426,-68.919902,0.000037,-69.739151,0.027417,-69.947758,0.000959,-69.606933
25%,90.493404,-25.838929,90.560375,-25.623512,90.337017,-26.073914,90.481895,-26.155572,90.051499,-25.892420
50%,180.013796,0.469332,179.988155,0.505898,179.997329,0.504364,180.004772,0.419967,179.990674,0.437977
75%,270.589278,25.901413,270.643138,25.633424,270.504069,26.082747,270.699466,26.266745,270.224740,26.120877
max,359.990157,69.288665,359.969741,68.886986,359.995370,69.724723,359.996740,69.905786,359.983575,69.534543


In [15]:
# Et voici comment concaténer avec les données précédemment collectées :
df_rte = pd.read_csv(input_datasets, parse_dates=['datetime_utc'])
df_joint = pd.merge(df_rte, df, on=['datetime_utc'])
df_joint.info()
display(df_joint)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106704 entries, 0 to 106703
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   datetime_utc  106704 non-null  datetime64[ns, UTC]
 1   consommation  106704 non-null  float64            
 2   solaire       106704 non-null  float64            
 3   tco_solaire   106704 non-null  float64            
 4   tch_solaire   106704 non-null  float64            
 5   cru_azimuth   106704 non-null  float64            
 6   cru_altitude  106704 non-null  float64            
 7   sel_azimuth   106704 non-null  float64            
 8   sel_altitude  106704 non-null  float64            
 9   svt_azimuth   106704 non-null  float64            
 10  svt_altitude  106704 non-null  float64            
 11  bra_azimuth   106704 non-null  float64            
 12  bra_altitude  106704 non-null  float64            
 13  eyg_azimuth   106704 non-null  float64      

,datetime_utc,consommation,solaire,tco_solaire,tch_solaire,cru_azimuth,cru_altitude,sel_azimuth,sel_altitude,svt_azimuth,svt_altitude,bra_azimuth,bra_altitude,eyg_azimuth,eyg_altitude
0,2019-12-31 23:00:00+00:00,6123.0,0.0,0.0,0.0,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,5907.0,0.0,0.0,0.0,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,5724.0,0.0,0.0,0.0,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,5749.0,0.0,0.0,0.0,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,5700.0,0.0,0.0,0.0,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106699,2026-01-31 20:30:00+00:00,5573.0,0.0,0.0,0.0,286.342039,-40.289762,286.967065,-40.389258,286.905693,-41.098655,285.941480,-40.540487,285.323519,-39.764508
106700,2026-01-31 21:00:00+00:00,5491.0,0.0,0.0,0.0,293.260657,-45.353013,293.971618,-45.394654,293.899339,-46.175371,292.800835,-45.669668,292.103434,-44.889023
106701,2026-01-31 21:30:00+00:00,5514.0,0.0,0.0,0.0,301.202197,-50.138700,302.002787,-50.113266,301.969264,-50.961570,300.701415,-50.525457,299.875272,-49.753083
106702,2026-01-31 22:00:00+00:00,5618.0,0.0,0.0,0.0,310.517465,-54.502032,311.400617,-54.398736,311.486274,-55.304997,310.013869,-54.961682,308.992088,-54.217459
